In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

file_path = '/content/POWER_Point_Daily_20260101_20260328_023d75N_079d20E_LST.csv (1).xls'


skip_rows = 0
with open(file_path, 'r') as f:
    for i, line in enumerate(f):
        if "-END HEADER-" in line:
            skip_rows = i + 1
            break


df = pd.read_csv(file_path, skiprows=skip_rows)


df.columns = [c.strip() for c in df.columns]

features_to_drop = ['YEAR', 'MO', 'DAY', 'DOY', 'YEAR', 'MONTH']
df_filtered = df.drop(columns=[c for c in features_to_drop if c in df.columns])


df_numeric = df_filtered.select_dtypes(include=[np.number]).dropna()

if not df_numeric.empty:
    
    X = df_numeric.iloc[:, :-1]
    y = df_numeric.iloc[:, -1]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    print(f'Successfully trained on {len(df_numeric)} rows.')
    print(f'Target variable: {df_numeric.columns[-1]}')
    print(f'Mean Squared Error: {mse}')
else:
    print("Still unable to parse numeric data. Please check if the file format matches expected NASA POWER output.")

In [ ]:

df_clean = df_numeric.replace(-999, np.nan).dropna()

X_clean = df_clean.iloc[:, :-1]
y_clean = df_clean.iloc[:, -1]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)


final_model = RandomForestRegressor(n_estimators=100, random_state=42)
final_model.fit(X_train_c, y_train_c)

print(f'Retrained on {len(df_clean)} clean rows.')
print(f'New MSE: {mean_squared_error(y_test_c, final_model.predict(X_test_c))}')

In [ ]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    
    features = np.array(data['features']).reshape(1, -1)
    prediction = final_model.predict(features)
    return jsonify({'prediction': prediction[0], 'target': 'GWETPROF'})


def run_app():
    app.run(port=5000)

threading.Thread(target=run_app).start()
print("API is running locally on port 5000. Use /predict endpoint.")